In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import requests
import re
import time
import random
import ast
import requests

In [2]:
# 反扒机制：
# 1.fake_useragent #
# 2.Cooki #用户登录
# 3.time.sleep(random.uniform()) #随机等待时间
# ##################################
# 4.r=requests.get(url+f'&_={int(datetime.now().timestamp()*1000)}',headers=self.headers) #在每个请求的URL后追加动态时间戳参数(重复请求同一URL时会用到)
# 5.分布式框架（如 Scrapy-Redis)
# 6.使用代理 IP

In [3]:
# 添加随机 User-Agent
USER_AGENTS = [
    # Chrome 浏览器
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/113.0.0.0 Safari/537.36",
    # Firefox 浏览器
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:114.0) Gecko/20100101 Firefox/114.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7; rv:113.0) Gecko/20100101 Firefox/113.0",
    # Safari 浏览器
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 11_2_3) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.0.3 Safari/605.1.15",
    "Mozilla/5.0 (iPhone; CPU iPhone OS 15_0 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/15.0 Mobile/15E148 Safari/604.1",
]


#获取HTML页面内容
def getHTMLText(url, retries=10):
    for attempt in range(retries):
        try:
            headers = {"User-Agent": random.choice(USER_AGENTS)}
            response = requests.get(url, headers=headers, timeout=30)
            response.raise_for_status()
            response.encoding = response.apparent_encoding
            return response.text
        except requests.exceptions.RequestException as e:
            print(f"获取页面出错：{e}。正在重试（第 {attempt + 1} 次，共 {retries} 次）...")
            if attempt < retries - 1:
                # 使用指数退避策略，动态调整等待时间
                time.sleep(2**attempt + random.uniform(1, 5))
            else:
                print(f"多次尝试 ({retries} 次) 后仍未能获取到 {url} 的数据。")
                return ""

            
            
    
#解析HTML页面并提取信息
def parseHTML(html):
    soup = BeautifulSoup(html, "lxml")
    # 提取名称
    h1_tag = soup.find("h1")
    weapon_name_full = h1_tag.text.strip()
    pattern = r'^(.*?)(\s\((.*?)\))$'  # 匹配武器名称和括号中的内容
    match = re.match(pattern, weapon_name_full)
    weapon_name = match.group(1).strip()
    condition = match.group(3).strip()
    

    # 提取品质信息
    detail_cont_div = soup.find("div", class_="market-list")
    p_tag = detail_cont_div.find("p") # 在该 <div> 下查找目标 <p> 标签
    spans = p_tag.find_all("span")
    span = spans[0]
    quality_text = span.get_text(strip=True)
    quality_text = quality_text.split('|')[-1].strip()
    

    # 提取不同外观对应的ID
    #普通_id: 
    bookmark_div = soup.find("div", class_="add-bookmark")
    target_id = bookmark_div["data-target-id"]
    
    relative_goods_div = soup.find("div", id="relative-goods")
    scope_btns = relative_goods_div.find("div", class_="scope-btns")
    buttons = scope_btns.find_all("a", class_="j_Goods-jump")
    wear_texts=[]
    goods_ids=[]
    wear_texts.append(condition)
    goods_ids.append(target_id)
    for button in buttons:
        wear_text=button.get_text(strip=True)
        goods_id=button.get("data-goodsid", "")
        if wear_text == 'StatTrak™':
            StatTrak_id = goods_id
        else:
            StatTrak_id = 0
        if wear_text and goods_id and 'StatTrak™' not in wear_text and '纪念品' not in wear_text:
            wear_texts.append(wear_text)
            goods_ids.append(goods_id)
    last_id = goods_ids[-1] #最后一个外观的id
            
    #StatTrak_id: 
    target_id_st=0
    wear_texts_st=[]
    goods_ids_st=[]
    if StatTrak_id:
        time.sleep(random.uniform(3,6))
        soup_st = BeautifulSoup(getHTMLText(f"https://buff.163.com/goods/{StatTrak_id}"), "lxml")
        bookmark_div_st = soup_st.find("div", class_="add-bookmark")
        target_id_st = bookmark_div_st["data-target-id"]
        wear_texts_st.append(condition)
        goods_ids_st.append(target_id_st)
        
        relative_goods_div_st = soup_st.find("div", id="relative-goods")
        scope_btns_st = relative_goods_div_st.find("div", class_="scope-btns")
        buttons_st = scope_btns_st.find_all("a", class_="j_Goods-jump")
        for button in buttons_st[:-1]:
            wear_text_st = button.get_text(strip=True)
            goods_id_st = button.get("data-goodsid", "")
            wear_texts_st.append(wear_text_st)
            goods_ids_st.append(goods_id_st)
        if len(goods_ids_st) != len(goods_ids):
            raise ValueError("解析错误")
         
    # 提取磨损区间
    pattern = r"paintwear_choices\s*:\s*(\[\[.*?\]\])"
    match = re.search(pattern, html, re.DOTALL)
    paintwear_choices = match.group(1)
    # 获取第一个数字
    first_number =ast.literal_eval(paintwear_choices)[0][0]

    time.sleep(random.uniform(3,6))
    html2 = getHTMLText(f"https://buff.163.com/goods/{last_id}")
    match = re.search(pattern, html2, re.DOTALL)
    paintwear_choices = match.group(1)
    # 获取最后一个数字
    last_number =ast.literal_eval(paintwear_choices)[-1][-1]
    
    # 确保输出不为 None
    assert (weapon_name, quality_text, wear_texts, goods_ids, wear_texts_st, goods_ids_st, first_number, last_number) is not None, "The result is None "
    return (weapon_name, quality_text, target_id_st, wear_texts, goods_ids, wear_texts_st, goods_ids_st, first_number, last_number)

# 配置 Selenium 选项以防止被检测
def configure_selenium():
    chrome_options = Options()
    chrome_options.add_argument("--headless")  # 无头模式
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")  # 禁用自动化标志
    chrome_options.add_argument("--start-maximized")
    chrome_options.add_argument("--no-sandbox")  #Chrome沙盒模式
    #chrome_options.add_argument("--disable-dev-shm-usage") #禁用共享内存使用
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"]) #排除 enable-automation 开关，避免浏览器向网站暴露自动化标志
    chrome_options.add_experimental_option("useAutomationExtension", False) #禁用 Selenium 的自动化扩展
    chrome_options.add_argument(f"user-agent={random.choice(USER_AGENTS)}")
    return chrome_options

In [4]:
# 输入武器箱中任一不是纪念品的饰品id
cases_and_collections={'画廊':'968134', '千瓦':'956469', '变革':'921561', '反冲':'900604', '梦魇':'886610','激流大行动':'871656', '蛇噬':'857542', '狂牙大行动':'835541', '裂空':'781636', 
               '棱彩2号':'779196', '裂网大行动':'775101', 'CS20':'773775', '棱彩':'769205', '头号特训':'763255', '地平线':'759247', '命悬一线':'45270', '光谱2号':'42342', '九头蛇大行动':'35479', '光谱':'42193', '手套武器箱':'35263', '伽玛2号':'34415', '伽玛':'36126', 
               '幻彩3号':'42027', '野火大行动':'35332', '左轮武器箱':'33946', '暗影':'42308', '弯曲猎手':'36145', '幻彩2号':'34915', '幻彩':'35355', '先锋':'34797', '电竞2014夏季':'42303', '突围':'35190', '猎杀者':'42201', '凤凰':'33960', '电竞2013冬季':'34811', 
               '冬季攻势':'35681', '军火交易3号':'39765', '英勇':'34955', '电竞2013':'36081', '军火交易2号':'42225', '军火交易':'38278', 
                       
               '狩猎运动':'968187', '平面设计':'968064', '2024死亡游乐园':'968078', '阿努比斯':'927292', '2021殒命大厦':'871507', '2021荒漠迷城':'871139', '2021炙热沙城Ⅱ':'871158', '2021列车停放站':'871134', '远古':'835438', '控制':'835335', '浩劫':'835366', '挪威人':'775823', '运河水城':'774785', 
               '圣马克镇':'775442', '2018炼狱小镇':'762212', '2018核子危机':'762142', '神魔':'35224', '解体厂':'34454', '旭日':'41972', '死城之谜':'34685', '死亡游乐园':'42217', '古堡激战':'42109', '行李仓库':'33924', '金库危机':'34836', '炙热沙城Ⅱ':'33964', '列车停放站':'36039', 
               '荒漠迷城':'35028', '意大利小镇':'35551', '湖畔激战':'42197', '安全处所':'36526', '无畏':'42288', '炙热沙城':'35971', '雨林遗迹':'34751', '殒命大厦':'33866', '炼狱小镇':'35337', '佣兵训练营':'36429', '核子危机':'35323', '办公室':'35622', '仓库突击':'35020', 
              }

values = list(cases_and_collections.values())
cases_and_collections_ids = [int(value) for value in values]

## 循环获取收藏箱中所有饰品信息

In [5]:
output_path = 'infodataaa.txt'
######################################################################
for cases_and_collections_id in cases_and_collections_ids[:]:#  index 
    max_retries=12
    retry_count=0
    while retry_count < max_retries:
        try:
            # 配置 Chrome 选项
            chrome_options = configure_selenium()
            # 创建浏览器驱动
            driver = webdriver.Chrome(options=chrome_options)
            # 设置窗口大小
            driver.set_window_size(1920, 1080)

            # 打开指定网页
            driver.get(f"https://buff.163.com/goods/{cases_and_collections_id}")
            time.sleep(random.uniform(3,6))
            
            # 注入 JavaScript 以隐藏 Selenium 的特性
            driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
                "source": """
                    Object.defineProperty(navigator, 'webdriver', {
                        get: () => undefined
                    });
                """
            })

            # 等待页面加载完成并找到需要点击的按钮
            wait = WebDriverWait(driver, random.uniform(5, 10))
            button = wait.until(EC.element_to_be_clickable((By.ID, "weapon_case_entry")))

            # 滚动到按钮位置
            driver.execute_script("arguments[0].scrollIntoView();", button)
            time.sleep(random.uniform(3,6))  # 等待滚动完成

            # 使用 JavaScript 点击按钮
            driver.execute_script("arguments[0].click();", button)

            # 等待新内容加载
            time.sleep(random.uniform(3, 6))

            # 获取页面内容
            html = driver.page_source

            # 使用 BeautifulSoup 解析 HTML
            soup = BeautifulSoup(html, "lxml")

            # 查找特定的 <ul class="weapon-list"> 标签
            weapon_list = soup.find("ul", class_="weapon-list")

            # 获取 <ul> 中的所有 <a> 标签
            buttons = weapon_list.find_all("a")
            current_case=buttons[0].text.strip()
            # 如果找到两个按钮，点击第二个按钮
            if len(buttons) == 2:
                current_case = buttons[1].text.strip()  # 获取第二个按钮的文本
                # 使用显式等待确保按钮可点击
                wait = WebDriverWait(driver, random.uniform(3, 5))
                second_button_element = wait.until(EC.element_to_be_clickable((By.LINK_TEXT, current_case)))
                # 使用 JavaScript 强制点击按钮
                driver.execute_script("arguments[0].scrollIntoView();", second_button_element)
                time.sleep(random.uniform(3, 6))  # 随机等待滚动完成
                driver.execute_script("arguments[0].click();", second_button_element)

            # 获取点击后更新的页面内容
            time.sleep(random.uniform(3, 6))
            html = driver.page_source

            # 使用 BeautifulSoup 解析更新后的 HTML
            soup = BeautifulSoup(html, "lxml")

            # 使用正则表达式从所有 <a> 标签中提取 "/goods/ID" 格式的链接
            pattern = re.compile(r'/goods/(\d+)')
            ids = []

            # 找到所有的 <a> 标签
            for a_tag in soup.find_all('a', href=True):
                if '收藏' not in a_tag.text and '武器' not in a_tag.text:
                    match = pattern.search(a_tag['href'])
                    if match:
                        ids.append(match.group(1))
            break
        except Exception as e:
            driver.quit() # 关闭浏览器
            retry_count += 1
            print(f"解析错误1: {e}. 正在重试...({retry_count}/{max_retries})")
            time.sleep(retry_count*30 + random.uniform(15, 30))
    driver.quit() # 关闭浏览器
    if retry_count == max_retries:
        # 中断后的索引
        key_to_find = str(cases_and_collections_id)
        if key_to_find in values:
            index = values.index(key_to_find)
        print("解析错误1多次尝试后已超时")
        break
##############################################
    with open(output_path, "a", encoding="utf-8") as file:
        file.write(f"收藏品：{current_case}\n\n")
    for item_id in ids:
        max_retries=12
        retry_count=0
        while retry_count < max_retries:
            try:
                html_content = getHTMLText(f"https://buff.163.com/goods/{item_id}")
                (weapon_name, quality_text, target_id_st, wear_texts, goods_ids, wear_texts_st, goods_ids_st, first_number, last_number) = parseHTML(html_content)
                break
            except Exception as e:
                retry_count += 1
                print(f"解析错误2: {e}. 正在重试...({retry_count}/{max_retries})")
                time.sleep(retry_count*10 + random.uniform(10, 30))

        if retry_count == max_retries:
            print("解析错误2多次尝试后已超时")
            with open(output_path, "a", encoding="utf-8") as file:
                file.write(f"从{item_id}开始缺失\n\n")
            break
        else:
            # 打开文件进行写入
            with open(output_path, "a", encoding="utf-8") as file:
                file.write(f"名称：{weapon_name}\n")
                file.write(f"品质：{quality_text}\n")
                file.write(f"磨损区间：{first_number}-{last_number}\n")
                file.write("普通：\n")
                for wear_text, goods_id in zip(wear_texts, goods_ids):
                    if "暂无在售" in wear_text:
                        wear_text = wear_text.replace("暂无在售", "").strip()
                    file.write(f"{wear_text}：{goods_id}\n")
                if target_id_st != 0:
                    file.write("StatTrak：\n")
                    for wear_text, goods_id in zip(wear_texts_st, goods_ids_st):
                        if "暂无在售" in wear_text:
                            wear_text = wear_text.replace("暂无在售", "").strip()
                        file.write(f"{wear_text}：{goods_id}\n")
                file.write("\n")
######################################################################


解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(2/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(1/12)
解析错误2: 'NoneType' object is not subscriptable. 正在重试...(1/12)
解析错误2: 'NoneType' object is not subscriptable. 正在重试...(2/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'text'. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'text'. 正在重试...(1/12)
解析错误2: 'NoneType' object is not subscriptable. 正在重试...(2/12)
解析错误2: 'NoneType' object has no attribute 'text'. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(1/12)
解析错误2: 'NoneType' object is not subscriptable. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(1/12)
解析错误2: 'NoneType' object is not subscriptabl

解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'text'. 正在重试...(1/12)
解析错误2: 'NoneType' object is not subscriptable. 正在重试...(2/12)
解析错误2: 'NoneType' object is not subscriptable. 正在重试...(1/12)
解析错误2: 'NoneType' object is not subscriptable. 正在重试...(1/12)
解析错误2: 'NoneType' object is not subscriptable. 正在重试...(2/12)
解析错误2: 'NoneType' object has no attribute 'text'. 正在重试...(1/12)
解析错误2: 'NoneType' object is not subscriptable. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(1/12)
解析错误2: 'NoneType' object is not subscriptable. 正在重试...(2/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'text'. 正在重试...(1/12)
解析错误2: 'NoneType' object is not subscriptable. 正在重试...(1/12)
解析错误2: 'NoneType' object is not subscriptable. 正在重试...(2/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(3

解析错误2: 'NoneType' object has no attribute 'text'. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'text'. 正在重试...(1/12)
解析错误2: 'NoneType' object is not subscriptable. 正在重试...(1/12)
解析错误2: 'NoneType' object is not subscriptable. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'text'. 正在重试...(2/12)
解析错误2: 'NoneType' object has no attribute 'text'. 正在重试...(1/12)
解析错误2: 'NoneType' object is not subscriptable. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'text'. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'text'. 正在重试...(1/12)
解析错误2: 'NoneType' object is not subscriptable. 正在重试...(2/12)
解析错误2: 'NoneType' object has no attribute 'text'. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(2/12)
解析错误2: 'NoneType' object is not subscriptable. 正在重试...(1/12)
解析错误2: 'NoneType' object is not subscriptable. 正在重试...(2/12)
解析错误2: 'NoneType' object has no attribute 'text'. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试..

解析错误1: Message: 
Stacktrace:
#0 0x563d30ec9403 <unknown>
#1 0x563d30ccf778 <unknown>
#2 0x563d30d06c87 <unknown>
#3 0x563d30d06e51 <unknown>
#4 0x563d30d39f24 <unknown>
#5 0x563d30d24a2d <unknown>
#6 0x563d30d37c74 <unknown>
#7 0x563d30d248f3 <unknown>
#8 0x563d30cfa0d8 <unknown>
#9 0x563d30cfb205 <unknown>
#10 0x563d30f10e3d <unknown>
#11 0x563d30f13db6 <unknown>
#12 0x563d30efa13e <unknown>
#13 0x563d30f149b5 <unknown>
#14 0x563d30eee970 <unknown>
#15 0x563d30f31228 <unknown>
#16 0x563d30f313bf <unknown>
#17 0x563d30f4babe <unknown>
#18 0x7f0c54b19ac3 <unknown>
. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'text'. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(2/12)
解析错误2: 'NoneType' object has no attribute 'text'. 正在重试...(1/12)
解析错误2: 'NoneType' object is not subscriptable. 正在重试...(2/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(1/12)
解析错误2: 'NoneType' object has n

解析错误2: 'NoneType' object is not subscriptable. 正在重试...(1/12)
解析错误2: 'NoneType' object is not subscriptable. 正在重试...(1/12)
解析错误2: 'NoneType' object is not subscriptable. 正在重试...(1/12)
解析错误2: 'NoneType' object is not subscriptable. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'text'. 正在重试...(2/12)
解析错误2: 'NoneType' object has no attribute 'text'. 正在重试...(3/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(4/12)
解析错误2: 'NoneType' object is not subscriptable. 正在重试...(5/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(6/12)
解析错误2: 'NoneType' object has no attribute 'text'. 正在重试...(7/12)
解析错误2: 'NoneType' object is not subscriptable. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'text'. 正在重试...(1/12)
解析错误2: 'NoneType' object is not subscriptable. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(2/12)
解析错误2: 'NoneType' object has no attribute 'text'. 正在重试...

解析错误1: Message: 
Stacktrace:
#0 0x56434f75a403 <unknown>
#1 0x56434f560778 <unknown>
#2 0x56434f597c87 <unknown>
#3 0x56434f597e51 <unknown>
#4 0x56434f5caf24 <unknown>
#5 0x56434f5b5a2d <unknown>
#6 0x56434f5c8c74 <unknown>
#7 0x56434f5b58f3 <unknown>
#8 0x56434f58b0d8 <unknown>
#9 0x56434f58c205 <unknown>
#10 0x56434f7a1e3d <unknown>
#11 0x56434f7a4db6 <unknown>
#12 0x56434f78b13e <unknown>
#13 0x56434f7a59b5 <unknown>
#14 0x56434f77f970 <unknown>
#15 0x56434f7c2228 <unknown>
#16 0x56434f7c23bf <unknown>
#17 0x56434f7dcabe <unknown>
#18 0x7fdf3cb21ac3 <unknown>
. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'text'. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'text'. 正在重试...(2/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(2/12)
解析错误2: 'NoneType' object h

解析错误2: 'NoneType' object has no attribute 'text'. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'text'. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(2/12)
解析错误2: 'NoneType' object has no attribute 'text'. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(2/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(3/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'text'. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(2/12)
解析错误2: 'NoneType' object has no attribute 'text'. 正在重试...(3/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'text'. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(2/12)
解析错误2: 'NoneType' object has no

解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(2/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(3/12)
解析错误2: 'NoneType' object has no attribute 'text'. 正在重试...(4/12)
解析错误2: 'NoneType' object has no attribute 'text'. 正在重试...(5/12)
解析错误2: 'NoneType' object has no attribute 'text'. 正在重试...(6/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(2/12)
解析错误2: 'NoneType' object has no attribute 'group'. 正在重试...(1/12)
解析错误2: 'NoneType' object has no attribute 'text'. 正在重试...(1/12)
解析错误1: Message: 
Stacktrace:
#0 0x55cf5ba21403 <unknown>
#1 0x55cf5b827778 <unknown>
#2 0x55cf5b85ec87 <unknown>
#3 0x55cf5b85ee51 <unknown>
#4 0x55cf5b891f24 <unknown>
#5 0x55cf5b87ca2d <unknown>
#6 0x55cf5b88fc74 <unknown>
#7 0x55cf5b87c8f3 <unknown>
#8 0x55cf5b8520d8 <unknown>
#9 0x55cf5b853205 <unknown>
#10 0x55cf5ba68e3d <unknown>
#11 0x55cf5ba6bdb6 <unknown>
#12 0x55cf5ba5213e <unknown>
#13 0x55cf5ba6c9b5 <unk